# 02 - Transform and Validate

Transform raw operational CSV files into data warehouse-shaped dimension and fact CSV files, then run data quality checks.

## Setup Project Paths

Prepare folder references for raw input, processed dim/fact output, and data quality output.

**Output:** `RAW_DIR`, `PROCESSED_DIR`, `OUTPUT_DIR`.

In [1]:
from pathlib import Path
from datetime import datetime
import calendar
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR = PROJECT_ROOT / 'output'
VALIDATION_DIR = PROJECT_ROOT / 'data' / 'validation'
for folder in [PROCESSED_DIR, OUTPUT_DIR, VALIDATION_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

## Read Raw Source Data

Load source CSV files from Notebook 01 and apply light preparation: parse dates and map missing transaction customers to `CUST-GUEST`.

**Input:** `data/raw/*.csv`.  
**Output:** source DataFrames ready for transformation.

In [2]:
customers = pd.read_csv(RAW_DIR / 'customers.csv', keep_default_na=False)
products = pd.read_csv(RAW_DIR / 'products.csv', keep_default_na=False)
stores = pd.read_csv(RAW_DIR / 'stores.csv', keep_default_na=False)
promotions = pd.read_csv(RAW_DIR / 'promotions.csv', keep_default_na=False)
promotions['promotion_type'] = promotions['promotion_type'].replace(['', 'None', 'nan', 'NaN'], 'No Promotion').fillna('No Promotion')
transactions = pd.read_csv(RAW_DIR / 'sales_transactions.csv', keep_default_na=False)
details = pd.read_csv(RAW_DIR / 'sales_details.csv', keep_default_na=False)
payments = pd.read_csv(RAW_DIR / 'payments.csv', keep_default_na=False)
inventory = pd.read_csv(RAW_DIR / 'inventory.csv', keep_default_na=False)
transactions['customer_id'] = transactions['customer_id'].fillna('').replace('', 'CUST-GUEST')
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])
inventory['snapshot_date'] = pd.to_datetime(inventory['snapshot_date'])
channels = transactions[['channel_id','channel_name','channel_type']].drop_duplicates().sort_values('channel_id')

## Build Dimension Tables

Create warehouse dimensions with surrogate keys while keeping original source IDs as business keys. `dim_time` is generated from transaction and inventory dates.

**Output:** `dim_time`, `dim_customer`, `dim_product`, `dim_store`, `dim_channel`, `dim_payment`, `dim_promotion`.

In [3]:
def make_dim_time(*date_series):
    dates = pd.concat([pd.to_datetime(s).dt.date for s in date_series]).dropna().drop_duplicates().sort_values()
    rows = []
    for full_date in dates:
        ts = pd.Timestamp(full_date)
        rows.append({'time_key':int(ts.strftime('%Y%m%d')),'full_date':ts.date(),'year':ts.year,'quarter':ts.quarter,'month':ts.month,'month_name':calendar.month_name[ts.month],'day':ts.day,'day_of_week':calendar.day_name[ts.weekday()],'is_weekend':ts.weekday() >= 5})
    return pd.DataFrame(rows)

dim_time = make_dim_time(transactions['transaction_date'], inventory['snapshot_date'])
dim_customer = customers[['customer_id','customer_name','gender','birth_date','loyalty_tier','customer_segment','province','city','registered_date']].drop_duplicates('customer_id').sort_values('customer_id').reset_index(drop=True); dim_customer.insert(0, 'customer_key', range(1, len(dim_customer) + 1))
dim_product = products[['product_id','product_name','brand','category','subcategory','unit_price','cost_price','launch_date','is_active']].drop_duplicates('product_id').sort_values('product_id').reset_index(drop=True); dim_product.insert(0, 'product_key', range(1, len(dim_product) + 1))
dim_store = stores[['store_id','store_name','store_type','province','city','region','open_date']].drop_duplicates('store_id').sort_values('store_id').reset_index(drop=True); dim_store.insert(0, 'store_key', range(1, len(dim_store) + 1))
dim_channel = channels[['channel_id','channel_name','channel_type']].drop_duplicates('channel_id').sort_values('channel_id').reset_index(drop=True); dim_channel.insert(0, 'channel_key', range(1, len(dim_channel) + 1))
dim_payment = payments[['payment_id','payment_method','payment_status','payment_provider']].drop_duplicates('payment_id').sort_values('payment_id').reset_index(drop=True); dim_payment.insert(0, 'payment_key', range(1, len(dim_payment) + 1))
dim_promotion = promotions[['promotion_id','promotion_name','promotion_type','discount_rate','start_date','end_date']].drop_duplicates('promotion_id').sort_values('promotion_id').reset_index(drop=True); dim_promotion.insert(0, 'promotion_key', range(1, len(dim_promotion) + 1))

## Build Fact Tables

Create `fact_sales` at line-item grain and `fact_inventory_snapshot` at product-store-date grain. Sales metrics are calculated here: gross sales, net sales, cost amount, and net profit.

**Output:** `fact_sales`, `fact_inventory_snapshot`.

In [4]:
sales = details.merge(transactions, on='transaction_id', how='left').merge(products[['product_id','cost_price']], on='product_id', how='left')
sales['time_key'] = sales['transaction_date'].dt.strftime('%Y%m%d').astype(int)
sales['gross_sales'] = sales['quantity'].astype(float) * sales['unit_price'].astype(float)
sales['discount_amount'] = sales['discount_amount'].fillna(0).astype(float)
sales['net_sales'] = sales['gross_sales'] - sales['discount_amount']
sales['cost_amount'] = sales['quantity'].astype(float) * sales['cost_price'].astype(float)
sales['net_profit'] = sales['net_sales'] - sales['cost_amount']
for dim, key_pair in [(dim_customer, ['customer_key','customer_id']), (dim_product, ['product_key','product_id']), (dim_store, ['store_key','store_id']), (dim_channel, ['channel_key','channel_id']), (dim_payment, ['payment_key','payment_id']), (dim_promotion, ['promotion_key','promotion_id'])]:
    sales = sales.merge(dim[key_pair], on=key_pair[1], how='left')
fact_sales = sales[['transaction_id','detail_id','time_key','customer_key','product_key','store_key','channel_key','payment_key','promotion_key','quantity','unit_price','gross_sales','discount_amount','net_sales','cost_amount','net_profit']].rename(columns={'quantity':'quantity_sold'}).sort_values(['transaction_id','detail_id']).reset_index(drop=True)
fact_sales.insert(0, 'sales_key', range(1, len(fact_sales) + 1))
inv = inventory.merge(dim_product[['product_key','product_id']], on='product_id', how='left').merge(dim_store[['store_key','store_id']], on='store_id', how='left')
inv['time_key'] = inv['snapshot_date'].dt.strftime('%Y%m%d').astype(int)
fact_inventory_snapshot = inv[['time_key','product_key','store_key','stock_quantity','reorder_level','stock_status']].sort_values(['store_key','product_key']).reset_index(drop=True)
fact_inventory_snapshot.insert(0, 'inventory_key', range(1, len(fact_inventory_snapshot) + 1))

## Export Processed Dim/Fact Files

Write all dimension and fact tables to `data/processed/` so the warehouse layer can be inspected before PostgreSQL loading.

**Output:** processed dim/fact CSV files.

In [5]:
processed_frames = {'dim_time.csv': dim_time, 'dim_customer.csv': dim_customer, 'dim_product.csv': dim_product, 'dim_store.csv': dim_store, 'dim_channel.csv': dim_channel, 'dim_payment.csv': dim_payment, 'dim_promotion.csv': dim_promotion, 'fact_sales.csv': fact_sales, 'fact_inventory_snapshot.csv': fact_inventory_snapshot}
for filename, df in processed_frames.items():
    df.to_csv(PROCESSED_DIR / filename, index=False)
    print(f'{filename}: {len(df):,} rows')

dim_time.csv: 513 rows
dim_customer.csv: 1,000 rows
dim_product.csv: 100 rows
dim_store.csv: 25 rows
dim_channel.csv: 3 rows
dim_payment.csv: 2,000 rows
dim_promotion.csv: 21 rows
fact_sales.csv: 4,688 rows
fact_inventory_snapshot.csv: 2,500 rows


## Run Data Quality Checks

Validate keys, numeric values, sales formulas, dimension mappings, guest customer handling, stock values, source-to-fact reconciliation, and payment statuses.

**Output:** `output/dq_summary.csv`.

## Data Validation Rules

Validasi transformasi disimpan sebagai file CSV di `data/validation/`, supaya hasil pengecekan bisa diaudit tanpa membuka notebook. File yang dibuat:

- `transform_table_validation.csv` untuk validasi jumlah row, kolom, dan duplicate key.
- `transform_numeric_validation.csv` untuk validasi quantity, price, sales, profit, dan stock.
- `transform_fk_validation.csv` untuk validasi foreign key dari fact ke dimension.
- `etl_summary_rejected_records.csv` untuk ringkasan record yang ditolak.
- `rejected_fact_sales_duplicate_detail.csv` dan `rejected_fact_sales_fk.csv` bila ada data bermasalah.

In [6]:
validation_frames = {
    'dim_time': dim_time,
    'dim_customer': dim_customer,
    'dim_product': dim_product,
    'dim_store': dim_store,
    'dim_channel': dim_channel,
    'dim_payment': dim_payment,
    'dim_promotion': dim_promotion,
    'fact_sales': fact_sales,
    'fact_inventory_snapshot': fact_inventory_snapshot,
}
key_columns = {
    'dim_time': 'time_key',
    'dim_customer': 'customer_key',
    'dim_product': 'product_key',
    'dim_store': 'store_key',
    'dim_channel': 'channel_key',
    'dim_payment': 'payment_key',
    'dim_promotion': 'promotion_key',
    'fact_sales': 'sales_key',
    'fact_inventory_snapshot': 'inventory_key',
}

table_rows = []
for table_name, df in validation_frames.items():
    key = key_columns[table_name]
    duplicate_key_count = int(df[key].duplicated().sum())
    table_rows.append({
        'table_name': table_name,
        'row_count': len(df),
        'column_count': len(df.columns),
        'primary_key': key,
        'duplicate_key_count': duplicate_key_count,
        'null_key_count': int(df[key].isna().sum()),
        'is_table_valid': duplicate_key_count == 0 and int(df[key].isna().sum()) == 0,
    })
transform_table_validation = pd.DataFrame(table_rows)
transform_table_validation.to_csv(VALIDATION_DIR / 'transform_table_validation.csv', index=False)

numeric_checks = [
    ('fact_sales', 'quantity_sold', 'value <= 0', int((fact_sales['quantity_sold'] <= 0).sum()), fact_sales.loc[fact_sales['quantity_sold'] <= 0, 'detail_id'].head(5).astype(str).tolist()),
    ('fact_sales', 'unit_price', 'value < 0', int((fact_sales['unit_price'] < 0).sum()), fact_sales.loc[fact_sales['unit_price'] < 0, 'detail_id'].head(5).astype(str).tolist()),
    ('fact_sales', 'gross_sales', 'gross_sales != quantity_sold * unit_price', int(((fact_sales['gross_sales'] - fact_sales['quantity_sold'] * fact_sales['unit_price']).abs() > 0.01).sum()), fact_sales.loc[((fact_sales['gross_sales'] - fact_sales['quantity_sold'] * fact_sales['unit_price']).abs() > 0.01), 'detail_id'].head(5).astype(str).tolist()),
    ('fact_sales', 'net_sales', 'value < 0', int((fact_sales['net_sales'] < 0).sum()), fact_sales.loc[fact_sales['net_sales'] < 0, 'detail_id'].head(5).astype(str).tolist()),
    ('fact_sales', 'net_profit', 'missing value', int(fact_sales['net_profit'].isna().sum()), fact_sales.loc[fact_sales['net_profit'].isna(), 'detail_id'].head(5).astype(str).tolist()),
    ('fact_inventory_snapshot', 'stock_quantity', 'value < 0', int((fact_inventory_snapshot['stock_quantity'] < 0).sum()), fact_inventory_snapshot.loc[fact_inventory_snapshot['stock_quantity'] < 0, 'inventory_key'].head(5).astype(str).tolist()),
]
transform_numeric_validation = pd.DataFrame([
    {
        'table_name': table,
        'column_name': column,
        'rule': rule,
        'invalid_row_count': count,
        'invalid_values_sample': ', '.join(sample),
        'is_numeric_valid': count == 0,
    }
    for table, column, rule, count, sample in numeric_checks
])
transform_numeric_validation.to_csv(VALIDATION_DIR / 'transform_numeric_validation.csv', index=False)

fk_specs = [
    ('fact_sales', 'time_key', dim_time, 'dim_time', 'time_key'),
    ('fact_sales', 'customer_key', dim_customer, 'dim_customer', 'customer_key'),
    ('fact_sales', 'product_key', dim_product, 'dim_product', 'product_key'),
    ('fact_sales', 'store_key', dim_store, 'dim_store', 'store_key'),
    ('fact_sales', 'channel_key', dim_channel, 'dim_channel', 'channel_key'),
    ('fact_sales', 'payment_key', dim_payment, 'dim_payment', 'payment_key'),
    ('fact_sales', 'promotion_key', dim_promotion, 'dim_promotion', 'promotion_key'),
    ('fact_inventory_snapshot', 'time_key', dim_time, 'dim_time', 'time_key'),
    ('fact_inventory_snapshot', 'product_key', dim_product, 'dim_product', 'product_key'),
    ('fact_inventory_snapshot', 'store_key', dim_store, 'dim_store', 'store_key'),
]
fact_lookup = {'fact_sales': fact_sales, 'fact_inventory_snapshot': fact_inventory_snapshot}
fk_rows = []
rejected_fk_frames = []
for fact_table, fact_column, dim_df, dimension_table, dimension_column in fk_specs:
    fact_df = fact_lookup[fact_table]
    valid_values = set(dim_df[dimension_column].dropna())
    invalid_mask = fact_df[fact_column].isna() | ~fact_df[fact_column].isin(valid_values)
    invalid_rows = fact_df.loc[invalid_mask].copy()
    if not invalid_rows.empty:
        invalid_rows['failed_fk_column'] = fact_column
        invalid_rows['expected_dimension'] = dimension_table
        rejected_fk_frames.append(invalid_rows)
    fk_rows.append({
        'fact_table': fact_table,
        'fact_column': fact_column,
        'dimension_table': dimension_table,
        'dimension_column': dimension_column,
        'invalid_row_count': int(invalid_mask.sum()),
        'invalid_values_sample': ', '.join(fact_df.loc[invalid_mask, fact_column].astype(str).head(5).tolist()),
        'is_fk_valid': int(invalid_mask.sum()) == 0,
    })
transform_fk_validation = pd.DataFrame(fk_rows)
transform_fk_validation.to_csv(VALIDATION_DIR / 'transform_fk_validation.csv', index=False)

rejected_duplicate_detail = fact_sales[fact_sales['detail_id'].duplicated(keep=False)]
rejected_duplicate_detail.to_csv(VALIDATION_DIR / 'rejected_fact_sales_duplicate_detail.csv', index=False)
if rejected_fk_frames:
    rejected_fact_sales_fk = pd.concat(rejected_fk_frames, ignore_index=True)
else:
    rejected_fact_sales_fk = pd.DataFrame(columns=list(fact_sales.columns) + ['failed_fk_column', 'expected_dimension'])
rejected_fact_sales_fk.to_csv(VALIDATION_DIR / 'rejected_fact_sales_fk.csv', index=False)

checks = [
    ('Duplicate transaction_id', int(transactions['transaction_id'].duplicated().sum()), 'critical'),
    ('Duplicate detail_id', int(details['detail_id'].duplicated().sum()), 'critical'),
    ('Quantity must be > 0', int((details['quantity'] <= 0).sum()), 'critical'),
    ('Unit price must be non-negative', int((details['unit_price'] < 0).sum()), 'critical'),
    ('Gross sales equals quantity * unit_price', int(((fact_sales['gross_sales'] - fact_sales['quantity_sold'] * fact_sales['unit_price']).abs() > 0.01).sum()), 'critical'),
    ('Net sales must be non-negative', int((fact_sales['net_sales'] < 0).sum()), 'critical'),
    ('Product maps to dim_product', int(fact_sales['product_key'].isna().sum()), 'critical'),
    ('Missing customer handled as Guest Customer', 0 if 'CUST-GUEST' in set(dim_customer['customer_id']) else 1, 'major'),
    ('Stock quantity must be non-negative', int((fact_inventory_snapshot['stock_quantity'] < 0).sum()), 'critical'),
    ('Source net sales equals fact_sales net sales', 0 if abs(((details['quantity'] * details['unit_price']) - details['discount_amount']).sum() - fact_sales['net_sales'].sum()) <= 1 else 1, 'critical'),
    ('Payment status must be valid', int((~payments['payment_status'].isin(['Paid','Pending','Failed','Refunded'])).sum()), 'major'),
]
dq_summary = pd.DataFrame(checks, columns=['check_name','failed_records','severity'])
dq_summary['status'] = dq_summary['failed_records'].apply(lambda x: 'PASS' if int(x) == 0 else 'FAIL')
dq_summary['checked_at'] = datetime.now().isoformat(timespec='seconds')
dq_summary.to_csv(OUTPUT_DIR / 'dq_summary.csv', index=False)
dq_summary.to_csv(VALIDATION_DIR / 'dq_summary.csv', index=False)

etl_summary_rejected_records = pd.DataFrame([
    {'rejected_file': 'rejected_fact_sales_duplicate_detail.csv', 'rejected_row_count': len(rejected_duplicate_detail), 'reason': 'Duplicate detail_id in fact_sales'},
    {'rejected_file': 'rejected_fact_sales_fk.csv', 'rejected_row_count': len(rejected_fact_sales_fk), 'reason': 'Invalid fact-to-dimension foreign key mapping'},
])
etl_summary_rejected_records.to_csv(VALIDATION_DIR / 'etl_summary_rejected_records.csv', index=False)

print('Validation files written to:', VALIDATION_DIR)
dq_summary

Validation files written to: D:\Semester 8\Data Warehouse\erajaya-data-warehouse\data\validation


,check_name,failed_records,severity,status,checked_at
0,Duplicate transaction_id,0,critical,PASS,2026-05-30T21:20:39
1,Duplicate detail_id,0,critical,PASS,2026-05-30T21:20:39
2,Quantity must be > 0,0,critical,PASS,2026-05-30T21:20:39
3,Unit price must be non-negative,0,critical,PASS,2026-05-30T21:20:39
4,Gross sales equals quantity * unit_price,0,critical,PASS,2026-05-30T21:20:39
5,Net sales must be non-negative,0,critical,PASS,2026-05-30T21:20:39
6,Product maps to dim_product,0,critical,PASS,2026-05-30T21:20:39
7,Missing customer handled as Guest Customer,0,major,PASS,2026-05-30T21:20:39
8,Stock quantity must be non-negative,0,critical,PASS,2026-05-30T21:20:39
9,Source net sales equals fact_sales net sales,0,critical,PASS,2026-05-30T21:20:39
